# Работа с данными

Прочитаем и изучим train_identity.csv и train_transaction.csv - это таблицы, на которых будем обучать и проверять модель.

In [4]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [5]:
transact_path = "../data/hard/raw/train_transaction.csv"
ident_path = "../data/hard/raw/train_identity.csv"

transact_df = pd.read_csv(transact_path)
ident_df = pd.read_csv(ident_path)

Таблицы соединены через TransactionID. Одна таблица содержит информацию об устройстве, а другая о самой транзакции. Соединим таблицы по ID.
Также таблицы имеют колонку TransactionDT - она отвечает за время. То есть данные имеют временной характер, отсортируем по времени.
Также убедимся, что датасет очень несбалансированный.

In [51]:
df = transact_df.merge(ident_df, on='TransactionID', how='left')
df = df.sort_values('TransactionDT')

print(f'Процент мошеннических операций: {((sum(df['isFraud']) / len(df['isFraud'])) * 100):.2f}%')

Процент мошеннических операций: 3.50%


*Пропуски*: если в колонце больше 80% пропусков, то удаляем это колонку. Также удалим заведомо ненужные колонки.

In [52]:
missing = df.isnull().mean().sort_values(ascending=False) #посчитали среднее число пропусков и отсортировали по убыванию
cols_to_drop = missing[missing > 0.8].index.tolist()
df = df.drop(columns=cols_to_drop)
df = df.drop(columns=['TransactionID', 'TransactionDT'])